# Gomoku Agent: tự học bằng self-play trên GPU của Kaggle

Notebook này chạy `loop.py` của repo [gomoku-agent](https://github.com/NguyenTienTuanHCMUS/gomoku-agent): model tự đánh với chính nó, học từ các ván đó, rồi đấu với bản tốt nhất; mạnh hơn thì thay.

**Cài đặt notebook (cột bên phải):**
- *Accelerator*: **GPU T4 x2** (script tự chia việc cho cả 2 GPU), hoặc P100.
- *Internet*: **On** (để tải mã nguồn từ GitHub).

**Cách chạy:** bấm **Save Version → Save & Run All (Commit)**. Notebook chạy nền tới 12 giờ, bạn có thể tắt máy. Script tự dừng trước giới hạn đó (`HOURS`).

**Kết quả** (tab *Output* của phiên bản đã chạy):
- `best_gomoku_pv.pt`: model tốt nhất. Chép về máy, ví dụ vào `models/pv2/gomoku_pv.pt`.
- `selfplay_output.zip`: toàn bộ tiến trình (ván tự đánh, model từng thế hệ, lịch sử).

**Chạy tiếp lần sau:** bấm *Add Input* → chọn output của chính notebook này (phiên bản trước) → chạy lại. Ô thứ 3 tự giải nén `selfplay_output.zip` và loop làm tiếp từ thế hệ đang dở.

In [ ]:
import glob, os, subprocess, shutil
REPO = "https://github.com/NguyenTienTuanHCMUS/gomoku-agent.git"
WORK = "/tmp/gomoku-agent"          # code + dữ liệu tạm; chỉ kết quả cuối được chép vào /kaggle/working
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "--depth", "1", REPO, WORK], check=True)
os.chdir(WORK)
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch; print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count())"

In [ ]:
# Chạy tiếp từ output của một phiên bản trước (nếu đã Add Input)
prev = glob.glob("/kaggle/input/**/selfplay_output.zip", recursive=True)
if prev:
    print("tiếp tục từ", prev[0])
    !unzip -o -q {prev[0]} -d {WORK}
    !cat runs/selfplay/state.json
else:
    print("bắt đầu mới từ models/pv2/gomoku_pv.pt")

In [ ]:
# Tham số chính
HOURS = 11      # Kaggle dừng phiên ở 12 giờ; loop không bắt đầu thế hệ mới nếu nó sẽ chạy quá mốc này
GENS  = 100     # số thế hệ tối đa (thường HOURS dừng trước)
GAMES = 2000    # số ván tự đánh mỗi thế hệ; xem dòng "games/hour" trong log rồi chỉnh cho mỗi thế hệ khoảng 1 giờ
SIMS  = 200     # lượt MCTS mỗi nước khi tự đánh

!python -u loop.py --gens {GENS} --hours {HOURS} --games {GAMES} --sims {SIMS}

In [ ]:
# Đóng gói kết quả vào /kaggle/working (tab Output)
import json
!zip -q -r /kaggle/working/selfplay_output.zip runs data/selfplay models/sp
state = json.load(open("runs/selfplay/state.json"))
shutil.copy(state["best"], "/kaggle/working/best_gomoku_pv.pt")
for h in state["history"]:
    print(f"gen {h['gen']:3d}: {h['score']:.3f} vs {h['vs']}  {'-> model tốt nhất mới' if h['promoted'] else ''}  ({h['minutes']} phút)")
print("model tốt nhất:", state["best"], "-> /kaggle/working/best_gomoku_pv.pt")